# Day 1: HuggingFace Inference -distilgpt2


## Environment & Imports

In [ ]:
!pip install transformers torch

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

## Model Loading and inference


In [ ]:
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

## Tokenize deep dive


In [29]:
prompt = "the future of cooking is"
inputs = tokenizer(prompt, return_tensors="pt")

In [30]:
print ("Input IDs:",inputs["input_ids"])
print("Tokens:",tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])) #Notice the `Ġ` prefix on tokens — that `Ġ` means "preceded by a space". The tokenizer doesn't split on words, it splits on subword units learned from frequency in training data.

Input IDs: tensor([[ 1169,  2003,   286, 10801,   318]])
Tokens: ['the', 'Ġfuture', 'Ġof', 'Ġcooking', 'Ġis']


## Generation Parameter Experiments

In [31]:
outputs = model.generate(
    inputs["input_ids"],
    attention_mask = inputs["attention_mask"],
    pad_token_id = tokenizer.eos_token_id,
    max_length = 50,
    do_sample=True,
    temperature=0.7,
    top_k=50,
    top_p=0.9
)

In [32]:
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

the future of cooking is already in our hands.”


## Observations & self experiments

### Baseline

In [33]:
outputs = model.generate(
    inputs["input_ids"],
    attention_mask = inputs["attention_mask"],
    pad_token_id = tokenizer.eos_token_id,
    max_length = 50,
    do_sample=True,
    temperature=1.0,
    top_k=50,
    top_p=1.0
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

the future of cooking is a serious debate on a subject that is in the mind of an enormous majority of people, that is not something that is going to be considered in the United States at this period," said Richard Moore, a professor of political science


### Low Temps

In [34]:

outputs = model.generate(
    inputs["input_ids"],
    attention_mask = inputs["attention_mask"],
    pad_token_id = tokenizer.eos_token_id,
    max_length = 50,
    do_sample=True,
    temperature=0.3,
    top_k=50,
    top_p=1.0
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

the future of cooking is the future of cooking.”


### High Temps

In [35]:
outputs = model.generate(
    inputs["input_ids"],
    attention_mask = inputs["attention_mask"],
    pad_token_id = tokenizer.eos_token_id,
    max_length = 50,
    do_sample=True,
    temperature=1.5,
    top_k=50,
    top_p=1.0
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

the future of cooking is always going to take the shape of food culture,• the very nature of life is changing′ . If you don. No-one wants more of this.


### Nucleus only

In [36]:
outputs = model.generate(
    inputs["input_ids"],
    attention_mask = inputs["attention_mask"],
    pad_token_id = tokenizer.eos_token_id,
    max_length = 50,
    do_sample=True,
    temperature=1.0,
    top_k=0,
    top_p=0.9
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

the future of cooking is dependent on your science. I use math to estimate the slope of the age of the egg paddy. The easiest way to determine the slope of the egg paddy is to tell you where the egg paddy gets most of


### Greedy decoding

In [38]:
# Experiment 5: Greedy decoding (no sampling)
outputs = model.generate(
    inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    pad_token_id=tokenizer.eos_token_id,
    max_length=50,
    do_sample=False  # greedy — always picks highest prob token
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

the future of cooking is a matter of time.”


In [47]:
outputs = model.generate(
    inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    repetition_penalty=1.2,
    pad_token_id=tokenizer.eos_token_id,
    max_length=50,
    do_sample=False  # greedy — always picks highest prob token
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

the future of cooking is a matter for the people.”


| Experiment      | Output quality        | Observation                          |
|---|---|---|
| Baseline (1.0)  | Coherent, wandered    | Balanced randomness                  |
| Low temp (0.3)  | Short + newlines      | Collapsed to high-prob tokens        |
| High temp (1.5) | More creative         | Riskier word choices                 |
| Nucleus (top_p) | Focused, coherent     | Cuts low-prob tail, keeps diversity  |
| Greedy | Focused, coherent     | Deterministic, same output every run |
| High rep penalty(1.2) | accurate, coherent| Slightly varied word choices at end|

# Things i wrote and learned today

### why do input_ids exist?
-> input_ids exist because they give a numerical   representation for the tokens which will be fed  into LM model
### what is the tokenizer actually doing and why cant you just pass raw text to the model?
-> tokenizer converts the input raw text into integers for the LM and they are called as tokens. we can't just pass raw text to the model since it is only able to understand the numerical representation of the raw text.

# Things i wrote today (Notes)
1. what a transformer model is used for?
transformers provides api and tools to easily download and train state of the artpre trained models
2. what a tokenizer is
tokenizers conver the input string into a list of integer tokens that would be input into the lm
3. how a text becomes tokens
the text are converted to a toke through the use of tokenizers
4. how inference works
it is the process where a trained machine learning model applies its learned patterns to new unseen data to make predictions, decision or generate outputs.
5.what are input_ids?
input_ids are the numerical representation of tokens which will be fed into the LM
->> Masked language models predict the probability of a word given surrounding word
->> causal language model predict the next word given a prefix sequence of words.

# **video ref =** http://youtube.com/watch?v=bZcKYiwtw1I